In [ ]:
# Cell 1: Setup
import os
import polars as pl
import spatial_split as ss

# Regions to process: set here, or via the REGIONS environment variable (comma separated)
REGIONS = os.environ.get('REGIONS', 'HIM,PEN,DES,IGP,WGR,ICR,NEI').split(',')
# REGIONS = ['DES']
STRESS_YEARS = {'HIM': [2003, 2001]}   # per region; regions not listed get the weakest unused usable year automatically
N_SPREAD = 8                           # evenly spread search years
plans = {}

In [2]:
# Cell 2: Usable years + search years per region (saved to data/study_years/{REGION}.json).
# Edit data/study_years/overrides.json to exclude years or fix the search years by hand, then rerun this cell.
for REGION in REGIONS:
    print(f"\n{'='*60}\n{REGION}\n{'='*60}")
    plans[REGION] = ss.plan_years(REGION, N_SPREAD, STRESS_YEARS.get(REGION))
    print(plans[REGION]['table'].to_pandas().to_string(index=False))


DES
[DES] usable 34 years (1991-2025); search 9: [1991, 1997, 2001, 2002, 2006, 2011, 2016, 2020, 2025]; stress 2002
 year   rows  polys  n_cls  min_cls  usable
 1987      0      0      0        0   False
 1988  56004    248     11        3   False
 1989  81889    350     11        4   False
 1990  91599    391     11        4   False
 1991 109196    506     11        4    True
 1992 111975    528     11        4    True
 1993 117308    558     11        4    True
 1994 117039    558     11        4    True
 1995  96519    428     11        4   False
 1996 116195    557     11        4    True
 1997 108138    518     11        4    True
 1998 117387    558     11        4    True
 1999 117344    558     11        4    True
 2000 117361    558     11        4    True
 2001 116969    555     11        4    True
 2002 109375    548     11        4    True
 2003 111133    556     11        4    True
 2004 116908    558     11        4    True
 2005 116969    558     11        4    True
 2

In [3]:
# Cell 3: What was chosen
for REGION, plan in plans.items():
    print(f"[{REGION}] usable {len(plan['usable'])}: {plan['usable']}")
    print(f"[{REGION}] search {len(plan['search'])}: {plan['search']} | stress year: {plan['stress']}")

[DES] usable 34: [1991, 1992, 1993, 1994, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
[DES] search 9: [1991, 1997, 2001, 2002, 2006, 2011, 2016, 2020, 2025] | stress year: 2002


In [4]:
# Cell 4: Build the polygon-level split (70:20:10) on the usable years. ~minutes per region.
# This run needs a NEW split (new fractions, polygon pinning for classes with < ss.MIN_POLYS polygons), so it is rebuilt by default.
# Set FORCE_SPLIT=0 to reuse an existing lookup instead (only once the new split is built).
FORCE = os.environ.get("FORCE_SPLIT", "1") == "1"
for REGION in REGIONS:
    print(f"\n--- {REGION} ---")
    ss.build_block_splits(REGION, force=FORCE)



--- DES ---
[DES] building split from 34 usable years, quality-gated
[DES] 558 polygons; pixels per polygon (median, p90, p99): 2300.0, 15048.2, 89270.8; extent km (median, p90, p99): 0.4, 1.5, 7.9
[DES] WARNING 0 keys hold more than one class and 10 span > 5 km: probably different polygons sharing one area_m2. See the polygon report.
[DES] 558 polygons in 544 centroid blocks of 0.01 deg, seed 174. Class report -> DES_poly_block0.01_s10000_y34_c68d15_class_report.csv
[DES] DATA GAP: classes with fewer than 10 polygons (low confidence; largest polygons in train, val/test from the smaller ones):
shape: (1, 5)
┌───────┬────────────┬──────────┬────────┬─────────┐
│ class ┆ n_polygons ┆ train_px ┆ val_px ┆ test_px │
│ ---   ┆ ---        ┆ ---      ┆ ---    ┆ ---     │
│ i64   ┆ i64        ┆ i64      ┆ i64    ┆ i64     │
╞═══════╪════════════╪══════════╪════════╪═════════╡
│ 61    ┆ 4          ┆ 129841   ┆ 20906  ┆ 16728   │
└───────┴────────────┴──────────┴────────┴─────────┘


In [5]:
# Cell 5: Check the split per year. Look for years where val/test miss a class, or are tiny.
for REGION in REGIONS:
    s = ss.split_summary(REGION)
    print(f"\n--- {REGION} ---")
    print(s.to_pandas().to_string(index=False))
    bad = s.filter((pl.col('val_missing') != '') | (pl.col('test_missing') != ''))
    print(f"{REGION}: {bad.height} of {s.height} years have a class missing from val or test")


--- DES ---
 year  train_px  train_polys  val_px  val_polys val_missing  test_px  test_polys test_missing
 1991     77258          242   21126        142                10812         122             
 1992     79000          250   21692        147                11283         131             
 1993     81417          263   24018        155                11873         140             
 1994     81311          263   23800        155                11928         140             
 1996     81439          263   22869        155                11887         139             
 1997     75410          246   22793        148                 9935         124             
 1998     81440          263   24019        155                11928         140             
 1999     81434          263   24019        155                11891         140             
 2000     81440          263   23993        155                11928         140             
 2001     81334          263   23857        153

In [6]:
# Cell 6: Polygon sanity check (same polygon -> one class; polygons stay whole in one split)
for REGION in REGIONS:
    print(f"\n--- {REGION} ---")
    ss.polygon_check(REGION, years=ss.study_years(REGION)[:3])


--- DES ---
year=1991: 117,387 rows, 0 without area_m2, 558 polygons, pixels/polygon median 68, p90 460
year=1992: 117,387 rows, 0 without area_m2, 558 polygons, pixels/polygon median 68, p90 460
year=1993: 117,387 rows, 0 without area_m2, 558 polygons, pixels/polygon median 68, p90 460

year=1991 vs year=1993: 0 shared system:index values (0.0% of year=1991); same lat/lon on 0; same area_m2 on 0
If shared == 0 the two years sample DIFFERENT pixels; if shared is large and lat/lon agree, the same pixels are reused.
polygon extent km (median, p90, p99, max): 0.46, 1.62, 7.97, 14.01;  keys with >1 class: 0


In [7]:
# Cell 7: What the later steps will use
for REGION in REGIONS:
    print(f"{REGION}: usable pool (split, 04 ranking, 06 pruning) {len(ss.study_years(REGION))} years | "
          f"search years (07 Optuna) {len(ss.search_years(REGION))}: {ss.search_years(REGION)}")

DES: usable pool (split, 04 ranking, 06 pruning) 34 years | search years (07 Optuna) 9: [1991, 1997, 2001, 2002, 2006, 2011, 2016, 2020, 2025]


In [8]:
# Cell 8: Data gaps per region = classes with fewer than ss.MIN_POLYS polygons. They are scored (ss.SCORE_FEW_POLY=True) but flagged
# low-confidence, and go into the per-region action items for the groups (collect more / larger reference polygons).
import pandas as pd
for REGION in REGIONS:
    scored, gaps = ss.scored_classes(REGION)
    print(f"{REGION}: {len(scored)} classes scored; low-confidence (< {ss.MIN_POLYS} polygons): {gaps if gaps else 'none'}")


DES: 11 classes scored; low-confidence (< 10 polygons): [61]
